In [1]:

import pandas as pd
from pathlib import Path

# ==========================================
# CONFIGURATION
# ==========================================

INPUT_FILE = Path("data/enriched/tanggal_enriched.xlsx")
OUTPUT_DIR = Path("output/reports")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

KEY = ["employee_id", "attendance_date"]

CHECK_COLUMNS = [
    "status",
    "shift",
    "clock_in_time",
    "clock_out_time",
    "attendance_in",
    "attendance_out",
    "data_source",
    "has_conflict"
]

# ==========================================
# 1. LOAD DATA
# ==========================================

df = pd.read_excel(INPUT_FILE)

df["employee_id"] = (
    df["employee_id"]
    .astype("string")
    .str.strip()
)

df["attendance_date"] = pd.to_datetime(
    df["attendance_date"],
    errors="coerce"
).dt.normalize()

print("Total rows:", len(df))
print("Total columns:", len(df.columns))

# ==========================================
# 2. EXACT DUPLICATES
# ==========================================

exact_mask = df.duplicated(
    keep=False
)

exact_duplicates = df.loc[exact_mask].copy()

print("\nExact duplicate rows:",
      len(exact_duplicates))

# ==========================================
# 3. DUPLICATE EMPLOYEE + DATE
# ==========================================

valid_keys = df[KEY].notna().all(axis=1)

duplicate_mask = (
    df.duplicated(subset=KEY, keep=False)
    & valid_keys
)

duplicates = df.loc[duplicate_mask].copy()

duplicates = duplicates.sort_values(KEY)

print("\nDuplicate employee-date rows:",
      len(duplicates))

print("Duplicate employee-date groups:",
      duplicates.groupby(KEY).ngroups)

# ==========================================
# 4. INSPECT CONFLICTS
# ==========================================

def unique_count(series):
    return series.astype("string").fillna(
        "<MISSING>"
    ).nunique()

available_checks = [
    col for col in CHECK_COLUMNS
    if col in duplicates.columns
]

summary = (
    duplicates.groupby(KEY, dropna=False)
    .agg(
        employee_name=(
            "employee_name", "first"
        ),
        record_count=(
            "employee_id", "size"
        ),
        **{
            f"{col}_unique": (col, unique_count)
            for col in available_checks
        }
    )
    .reset_index()
)

for col in available_checks:
    summary[f"{col}_different"] = (
        summary[f"{col}_unique"] > 1
    )

difference_columns = [
    f"{col}_different"
    for col in available_checks
]

summary["has_difference"] = (
    summary[difference_columns].any(axis=1)
)

summary["difference_count"] = (
    summary[difference_columns].sum(axis=1)
)

summary = summary.sort_values(
    ["difference_count", "record_count"],
    ascending=False
)

# ==========================================
# 5. INSPECT ONE EMPLOYEE
# ==========================================

if not summary.empty:
    example = summary.iloc[0]

    employee_records = duplicates.loc[
        (duplicates["employee_id"] == example["employee_id"])
        & (
            duplicates["attendance_date"]
            == example["attendance_date"]
        )
    ]

    display_columns = [
        col for col in
        KEY + ["employee_name"] + CHECK_COLUMNS
        if col in employee_records.columns
    ]

    print("\nExample duplicate:")
    print(
        employee_records[display_columns]
        .to_string(index=False)
    )

# ==========================================
# 6. EXPORT REPORTS
# ==========================================

with pd.ExcelWriter(
    OUTPUT_DIR / "duplicate_inspection.xlsx"
) as writer:

    summary.to_excel(
        writer,
        sheet_name="Duplicate Summary",
        index=False
    )

    duplicates.to_excel(
        writer,
        sheet_name="Duplicate Records",
        index=False
    )

    exact_duplicates.to_excel(
        writer,
        sheet_name="Exact Duplicates",
        index=False
    )

    df.loc[~valid_keys].to_excel(
        writer,
        sheet_name="Missing Keys",
        index=False
    )

print("\nReport created:")
print(OUTPUT_DIR / "enriched_duplicate_inspection.xlsx")


Total rows: 171605
Total columns: 57

Exact duplicate rows: 0

Duplicate employee-date rows: 232
Duplicate employee-date groups: 116

Example duplicate:
employee_id attendance_date employee_name     status            shift clock_in_time clock_out_time attendance_in attendance_out    data_source  has_conflict
  682609039      2026-10-02         TIRAN Incomplete              NaN         13:53          22:14           NaN            NaN tanggal + cuti         False
  682609039      2026-10-02         TIRAN     Posted SN219 - SHIFT219         14:00          22:00   HDR - HADIR            NaN tanggal + cuti         False

Report created:
output\reports\enriched_duplicate_inspection.xlsx


In [2]:

import pandas as pd

df = pd.read_excel("data/enriched/tanggal_enriched.xlsx")

# Identify duplicates by employee and attendance date
KEY = ["employee_id", "attendance_date"]

# Select ALL rows belonging to duplicate groups
duplicates = df[
    df.duplicated(subset=KEY, keep=False)
].copy()

# Sort so duplicated records appear next to each other
duplicates = duplicates.sort_values(KEY)

# Show every column and row without truncation
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(duplicates)

print("Total duplicate rows:", len(duplicates))
print("Total duplicate groups:", duplicates.groupby(KEY).ngroups)

duplicates.to_csv(
    "output/reports/enriched_duplicate_inspection.",
    index=False
)

,employee_name,employee_id,attendance_date,day_s,shift,shift_in,shift_out,clock_in_time,clock_out_time,next_day_clock_time,clock_in_time_be,clock_out_time_be,attendance_in,attendance_out,total_work_hours,attendance_type,work_group,enter_overtime_awh,exit_overtime_awh,next_day_oawh,total_overtime_hours_awf,enter_overtime_bwh,exit_overtime_bwf,next_day_bwh,total_overtime_hours_bwh,total_overtime_hours,automatic_overtime,spl_no,status,attendance_source,attendance_edit_date,attendance_description,check_in_attendance_location_mob,check_out_attendance_location_mob,company_code,company_name,location_code,location_name,job_title_code,job_title_name,organization_code,organization_name,organization_group_code,organization_group_name,cost_center_code,cost_center_name,job_level_code,job_level_name,employment_status,organization,job_title,location,last_update,updated_by,source_file,data_source,has_conflict
108040,TIRAN,682609039,2026-10-02,Friday,NaN,NaN,NaN,13:53,22:14,NaN,13:53,22:14,NaN,NaN,8 Hour(s) 21 Minute(s),Present,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,Incomplete,Import From Attendance,NaT,NaN,NaN,NaN,MPPA,PT Matahari Putra Prima Tbk,339,339-HYPERMART PALOPO,HPM339MT03,MITRA CLEANING,HP339.BACKEND,BACK END,339,SO HPM 339,33972023,339-SPV & STAFF RECEIVING CHECKING,1A,STAFF,MITRA,HP339.BACKEND - BACK END,HPM339MT03 - MITRA CLEANING,339 - 339-HYPERMART PALOPO,2026-10-15 11:44:34,Services,1-17 Oct 26_cleaned.xlsx,tanggal + cuti,False
108041,TIRAN,682609039,2026-10-02,Friday,SN219 - SHIFT219,14:00,22:00,14:00,22:00,NaN,14:00,22:00,HDR - HADIR,NaN,8 Hour(s) 0 Minute(s),Present,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,Posted,Upload from Excel,NaT,NaN,NaN,NaN,MPPA,PT Matahari Putra Prima Tbk,339,339-HYPERMART PALOPO,HPM339MT03,MITRA CLEANING,HP339.BACKEND,BACK END,339,SO HPM 339,33972023,339-SPV & STAFF RECEIVING CHECKING,1A,STAFF,MITRA,HP339.BACKEND - BACK END,HPM339MT03 - MITRA CLEANING,339 - 339-HYPERMART PALOPO,2026-10-15 15:21:58,HRISSGM339,1-17 Oct 26_cleaned.xlsx,tanggal + cuti,False
108042,TIRAN,682609039,2026-10-03,Saturday,NaN,NaN,NaN,13:56,NaN,NaN,13:56,NaN,NaN,NaN,NaN,Present,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,Incomplete,Import From Attendance,NaT,NaN,NaN,NaN,MPPA,PT Matahari Putra Prima Tbk,339,339-HYPERMART PALOPO,HPM339MT03,MITRA CLEANING,HP339.BACKEND,BACK END,339,SO HPM 339,33972023,339-SPV & STAFF RECEIVING CHECKING,1A,STAFF,MITRA,HP339.BACKEND - BACK END,HPM339MT03 - MITRA CLEANING,339 - 339-HYPERMART PALOPO,2026-10-15 11:44:34,Services,1-17 Oct 26_cleaned.xlsx,tanggal + cuti,False
108043,TIRAN,682609039,2026-10-03,Saturday,SN219 - SHIFT219,14:00,22:00,14:00,22:00,NaN,14:00,22:00,HDR - HADIR,NaN,8 Hour(s) 0 Minute(s),Present,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,Posted,Upload from Excel,NaT,NaN,NaN,NaN,MPPA,PT Matahari Putra Prima Tbk,339,339-HYPERMART PALOPO,HPM339MT03,MITRA CLEANING,HP339.BACKEND,BACK END,339,SO HPM 339,33972023,339-SPV & STAFF RECEIVING CHECKING,1A,STAFF,MITRA,HP339.BACKEND - BACK END,HPM339MT03 - MITRA CLEANING,339 - 339-HYPERMART PALOPO,2026-10-15 15:21:58,HRISSGM339,1-17 Oct 26_cleaned.xlsx,tanggal + cuti,False
108044,TIRAN,682609039,2026-10-04,Sunday,NaN,NaN,NaN,13:52,22:26,NaN,13:52,22:26,NaN,NaN,8 Hour(s) 34 Minute(s),Present,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,Incomplete,Import From Attendance,NaT,NaN,NaN,NaN,MPPA,PT Matahari Putra Prima Tbk,339,339-HYPERMART PALOPO,HPM339MT03,MITRA CLEANING,HP339.BACKEND,BACK END,339,SO HPM 339,33972023,339-SPV & STAFF RECEIVING CHECKING,1A,STAFF,MITRA,HP339.BACKEND - BACK END,HPM339MT03 - MITRA CLEANING,339 - 339-HYPERMART PALOPO,2026-10-15 11:44:34,Services,1-17 Oct 26_cleaned.xlsx,tanggal + cuti,False
108045,TIRAN,682609039,2026-10-04,Sunday,SN219 - SHIFT219,14:00,22:00,14:00,22:00,NaN,14:00,22:00,HDR - HADIR,NaN,8 Hour(s) 0 Minute(s),Present,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,Posted,Upload from Excel,NaT,NaN,NaN,NaN,MPPA,PT Matahari Putra Prima Tbk,339,339-HYPERMART PALOPO,HPM33

Total duplicate rows: 232
Total duplicate groups: 116
